In [ ]:
prefix_path = '../..'
import sys
sys.path.append(prefix_path)

import os
import gc
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

from detection_baselines.detection_halp import load_halp_dataset, extract_halp
from egh_vlm.utils import Qwen3ModelBundle, load_phd_dataset

In [ ]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3_vl_2b.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_PATH = f'{prefix_path}/data/phd/features/halp_phd_qwen3_layer24.pt'

#### Set Up

In [ ]:
dataset = load_phd_dataset(
    dataset_path=DATASET_PATH,
    img_folder_path=IMG_FOLDER_PATH)

features = load_halp_dataset(FEATURES_PATH) if os.path.isfile(FEATURES_PATH) else None
print(f'Length of processed features: {len(features) if features is not None else 0}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    dtype=torch.float16,
    device_map=device
)
print('config.dtype:', model.config.dtype)
processor = AutoProcessor.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    max_pixels=1024 * 1024)
model_bundle = Qwen3ModelBundle(model, processor, device)

#### Experiment

In [ ]:
features = extract_halp(
    dataset=dataset,
    model_bundle=model_bundle,
    client_type='qwen3',
    save_path=FEATURES_PATH,
    mask_mode=None,
    targeted_layer=23  # Layer 24
)
if features is not None and len(features) > 0:
    print('Shape of hidden state:', features.hiddens[0].shape)

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()